# Network Programming in Python
* this notebook introduces core network programming ideas in Python, moving from basic protocol concepts to practical HTTP clients and basic socket-based applications

The big idea is this:

> network programming is about moving data between programs, usually by following a protocol and translating between Python objects, text, and bytes


## Network Programming Concepts and Protocol Layers
* network programming usually involves several layers of abstraction
* a simplified view looks like this:
  * application layer
    * HTTP, SMTP, DNS and other application protocols
  * transport layer
    * TCP or UDP
  * internet layer
    * IP addressing and routing
* Python network code is typically written at the application layer, but it helps to know what is happening underneath

## TCP vs. UDP, IP, Ports, and Sockets
* key terms
  * **IP** (internet protocol)
    * identifies a machine on a network
  * **port**
    * identifies a service or endpoint on that machine
  * **socket**
    * a programming object representing one endpoint of communication
  * **TCP** (transmission control protocol)
    * connection-oriented
    * reliable, ordered delivery
    * common for HTTP, SMTP, databases, APIs
  * **UDP** (user datagram protocol)
    * connectionless
    * lower overhead
    * no delivery guarantees
    * common for streaming, games, DNS-style use cases
  * a useful way to think of TCP vs. UDP:
    * use TCP when correctness and ordered delivery matter
    * use UDP when speed and low overhead matter more than guaranteed delivery

## Client-Server Model

Much of network programming follows a client-server model

* the **server** listens for connections or requests
* the **client** connects and sends requests

Examples:

* browser → web server
* app → API server
* email client → SMTP server


## Request / Response Pattern
* many network interactions follow this pattern:
  * client sends a request
  * server processes it
  * server sends back a response
* HTTP is the classic example
* this pattern is often easier to reason about than long-running bidirectional protocols

## Stateless vs. Stateful Protocols
* a **stateless** protocol does not require the server to remember previous requests
* a **stateful** protocol depends on ongoing context between messages
* examples:
  * HTTP is often treated as stateless
  * a long-running socket conversation is often stateful (i.e., each message depends on what happened earlier)
  * SMTP sessions have state across commands
  * this matters because stateful protocols usually require more coordination and bookkeeping


## Python Support for Network Programming
* Python has several important networking-related modules and libraries:
* __`socket`__
  * low-level networking interface
* __`http`__
  * HTTP-related modules in the standard library
* __`urllib.request`__
  * standard-library URL fetching
* __`smtplib`__
  * SMTP client for sending email
* __`requests`__
  * third-party HTTP client that is usually easier to use in practice

## Text vs. Binary Data
* network communication ultimately moves **bytes**, not Python strings
* that means we often need to translate between:
  * __`str`__ and __`bytes`__
  * text formats and binary formats
* a common pattern is:
  * encode text before sending
  * decode bytes after receiving

In [2]:
message = 'hello'
data = message.encode('utf-8')

print(message)
print(data)
print(type(message))
print(type(data))

decoded = data.decode('utf-8')
print(decoded)

hello
b'hello'
<class 'str'>
<class 'bytes'>
hello


### Encoding and Decoding
* encoding turns text into bytes
* decoding turns bytes back into text
* UTF-8 is the most common text encoding used on the network

In [1]:
text = 'Boulder'
encoded = text.encode('utf-8')
decoded = encoded.decode('utf-8')

print(encoded)
print(decoded)


b'Boulder'
Boulder


## Fetching Web Resources with __`urllib.request`__
* the standard library includes __`urllib.request`__ for opening URLs
* useful to know because it is always available

In [3]:
from urllib.request import urlopen

# URL of a simple JSON test API
url = 'https://jsonplaceholder.typicode.com/todos/1'

# open the URL and wait for the server response
# urlopen() returns bytes, not a Python string
# timeout=10 means give up after 10 seconds if no response
with urlopen(url, timeout=10) as response:

    # read the raw bytes from the response body
    raw = response.read()

    # convert bytes -> string using UTF-8 decoding
    text = raw.decode('utf-8')

# verify that the original data was bytes
print(type(raw))

# print just the first 100 characters of the text response
print(text[:100])

<class 'bytes'>
{
  "userId": 1,
  "id": 1,
  "title": "delectus aut autem",
  "completed": false
}


If you are working with JSON, you often decode the bytes and then parse the text


In [4]:
from urllib.request import urlopen
import json

# URL of a simple JSON test API
url = 'https://jsonplaceholder.typicode.com/todos/1'

# open the URL and wait for the response
with urlopen(url, timeout=10) as response:

    # response.read() gives raw bytes
    # decode('utf-8') converts bytes -> string
    # json.loads() converts JSON text -> Python dictionary
    data = json.loads(response.read().decode('utf-8'))

# print the full Python dictionary
print(data)

# access one specific field from the JSON response
print(data['title'])

{'userId': 1, 'id': 1, 'title': 'delectus aut autem', 'completed': False}
delectus aut autem


## __`urllib.request`__ vs. __`requests`__
* it is useful to know both, but they play different roles
  * __`urllib.request`__
    * standard library
    * lower-level and more verbose
    * good to know for foundations
  * __`requests`__
    * third-party library
    * cleaner API
    * usually what you will use in practice for ordinary HTTP client code
* (for most day-to-day API work, __`requests`__ is usually simpler and more readable)


## Basic GET Requests with __`requests`__
* the __`requests`__ library makes HTTP requests feel much more direct

In [ ]:
import requests

# send an HTTP GET request to the URL
# timeout=10 means stop waiting after 10 seconds
response = requests.get(
    'https://jsonplaceholder.typicode.com/todos/1',
    timeout=10
)

# HTTP status code (200 = success)
print(response.status_code)

# response body as text (string)
# show only the first 100 characters
print(response.text[:100])

# automatically parse JSON into a Python dictionary
print(response.json())

### Query Parameters and Headers

HTTP requests often include query parameters and headers


In [5]:
import requests

# endpoint that returns information about the request we sent
url = 'https://httpbin.org/get'

# query parameters added to the URL
# this becomes something like:
# ?city=Boulder&units=imperial
params = {
    'city': 'Boulder',
    'units': 'imperial'
}

# HTTP headers sent with the request
# User-Agent identifies the client making the request
headers = {
    'User-Agent': 'Intermediate-Python-Notebook'
}

# send the GET request with params and headers
response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=10
)

# HTTP status code (200 = success)
print(response.status_code)

# parse the JSON response into a Python dictionary
print(response.json())

200
{'args': {'city': 'Boulder', 'units': 'imperial'}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate, zstd', 'Host': 'httpbin.org', 'User-Agent': 'Intermediate-Python-Notebook', 'X-Amzn-Trace-Id': 'Root=1-69e9922b-32eb7c54131a286909198211'}, 'origin': '66.33.8.95', 'url': 'https://httpbin.org/get?city=Boulder&units=imperial'}


### Handling Responses
* common response attributes and methods include:
  * __`status_code`__
  * __`text`__
  * __`json()`__
* these cover a large percentage of everyday API client work


## Building a Basic JSON REST Client
* a basic REST client often means:
  * make an HTTP request
  * parse JSON
  * extract the fields you care about
  * handle errors cleanly
* note that REST often maps CRUD operations to HTTP methods:

  Create, Read, Update, Delete → POST, GET, PUT, DELETE

In [7]:
import requests

def fetch_todo(todo_id):
    # build the URL using the todo ID
    url = f'https://jsonplaceholder.typicode.com/todos/{todo_id}'

    # send the GET request
    response = requests.get(url, timeout=10)

    # raise an exception if the request failed
    # for example: 404 or 500 errors
    response.raise_for_status()

    # convert JSON response -> Python dictionary
    data = response.json()

    # return only the fields we care about
    return {
        'id': data['id'],
        'title': data['title'],
        'completed': data['completed'],
    }

# call the function and print the result
print(fetch_todo(1))

{'id': 1, 'title': 'delectus aut autem', 'completed': False}


## Sending Email with __`smtplib`__
* Python includes __`smtplib`__ for talking to SMTP servers
* this is different from using __`requests`__ to call a web-based email service API
* useful distinction:
  * __`smtplib`__ speaks SMTP directly
  * __`requests`__ speaks HTTP to a service that may send email on your behalf

In [6]:
import smtplib
from email.message import EmailMessage

message = EmailMessage()
message['Subject'] = 'Test message'
message['From'] = 'sender@example.com'
message['To'] = 'recipient@example.com'
message.set_content('Hello from Python')

# we can run an SMTP server locally...
# python -m aiosmtpd -n -l localhost:1025

with smtplib.SMTP('localhost', 1025) as server:
    server.send_message(message)

## Sockets and Low-Level Networking
* the __`socket`__ module gives you low-level control over network communication
* you work directly with:
  * addresses
  * ports
  * connections
  * bytes
* this is lower-level than HTTP clients like __`requests`__, but it helps explain how client-server communication works underneath higher-level protocols

## Basic Socket Server
* this example shows a tiny TCP server that accepts one connection, receives bytes, and sends a response back
* run the server first, then run the client in a separate process or terminal

In [ ]:
import socket

# localhost (this machine)
HOST = '127.0.0.1'

# port number the server will listen on
PORT = 65432

# create a TCP socket
# AF_INET = IPv4
# SOCK_STREAM = TCP
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server:

    # attach the socket to the host and port
    server.bind((HOST, PORT))

    # start listening for incoming client connections
    server.listen()

    print(f'server listening on {HOST}:{PORT}')

    # wait (block) until a client connects
    # conn = new socket for this specific client
    # addr = address of the connected client
    conn, addr = server.accept()

    # work with the connected client
    with conn:
        print('connected by', addr)

        # receive up to 1024 bytes from the client
        data = conn.recv(1024)

        print('received bytes:', data)

        # send bytes back to the client
        # b'' means this is raw bytes, not a string
        conn.sendall(b'Hello from server')

## Basic Socket Client
* this client connects to the server, sends bytes, and reads the reply

In [ ]:
import socket

# localhost (this machine)
HOST = '127.0.0.1'

# port number the server is listening on
PORT = 65432

# create a TCP client socket
# AF_INET = IPv4
# SOCK_STREAM = TCP
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client:

    # connect to the server using its host and port
    # this must match the server's bind() values
    client.connect((HOST, PORT))

    # send a message to the server
    # sockets send bytes, so convert string -> bytes with encode()
    client.sendall('Hello from client'.encode('utf-8'))

    # wait for a reply from the server
    # receive up to 1024 bytes
    data = client.recv(1024)

# convert bytes -> string using decode()
print('received from server:', data.decode('utf-8'))

This example highlights a few essentials:

* sockets send and receive bytes
* clients connect to servers at an IP and port
* application-level meaning is built on top of those bytes


## A Tiny JSON Socket Example
* common pattern:
  * convert Python data to JSON text
  * encode JSON text to bytes
  * send the bytes
  * decode and parse on the other side

* that gives you a simple application protocol


In [8]:
import json

# regular Python dictionary
payload = {
    'city': 'Boulder',
    'temp': 72
}

# convert Python dict -> JSON text with json.dumps()
# then convert string -> bytes with encode('utf-8')
# this is a common pattern before sending data over a network
raw = json.dumps(payload).encode('utf-8')

# show the raw bytes that would be sent
print(raw)

# convert bytes -> string with decode('utf-8')
# then convert JSON text -> Python dict with json.loads()
decoded = json.loads(raw.decode('utf-8'))

# show the reconstructed Python dictionary
print(decoded)

b'{"city": "Boulder", "temp": 72}'
{'city': 'Boulder', 'temp': 72}


## Practical Guidance
* a few practical rules go a long way:
  * use __`requests`__ for most ordinary HTTP client work
  * know __`urllib.request`__ because it is standard library and foundational
  * know __`socket`__ to understand lower-level networking
  * use __`smtplib`__ when you need to speak SMTP directly
  * be careful about text vs. bytes
  * always think about timeouts and error handling in network code

## Mini Exercise
* write a small function that
  * accepts a URL
  * fetches it with __`requests`__
  * returns the HTTP status code and the parsed JSON
* then answer:
  * what happens if the response is not JSON?
  * where do bytes become text?

## Summary
* key ideas from this notebook:
  * network programming moves bytes between programs using protocols
  * TCP and UDP solve different transport problems
  * sockets are low-level communication endpoints
  * client-server and request-response are foundational patterns
  * HTTP is usually easier to work with than raw sockets
  * __`urllib.request`__ is standard-library URL access
  * __`requests`__ is usually the practical HTTP client for day-to-day work
  * __`smtplib`__ is for speaking SMTP directly
  * encoding and decoding matter because the network deals in bytes
  * sockets help you understand what higher-level libraries are doing for you

The goal is not to memorize every API!

But rather, to understand what layer you are working at and choose the right tool for that layer...
